In [1]:
import pandas as pd

shifaa = pd.read_csv("data/shifaa-train.csv")
# exclude non-mental health questions
shifaa = shifaa[~shifaa["Hierarchical Diagnosis"].str.contains("أمراض الغدد والهرمونات - الهرمونات وأثر اضطرابها")]
print(f"Number of unique questions: {shifaa['Question'].nunique()}")

Number of unique questions: 2096


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.8-27B", device_map="auto", dtype=torch.bfloat16, low_cpu_mem_usage=True)

/gpfs/automountdir/gpfs/homes/SEAS/home/g21775526/code/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

PROMPT = """
أنت مساعد طبيب نفسي عربي. سأعطيك سؤالاً، وسوف تقدم إجابة مفصلة باللغة العربية. إذا كان السؤال لا يتعلق بالصحة النفسية، سترد بـ "آسف، يمكنني فقط الإجابة على الأسئلة المتعلقة بالصحة النفسية."

مثال:
السؤال: {shot}
الإجابة: {shot_answer}

السؤال: {question}
"""

# Get one example for few-shot (use same sample consistently)
sample = shifaa.sample(1).iloc[0]
shot = sample["Question"]
shot_answer = sample["Answer"]

# Prepare all prompts
questions = shifaa["Question"].tolist()
prompts = [PROMPT.format(shot=shot, shot_answer=shot_answer, question=q) for q in questions]

In [ ]:
# ask the model in batch mode
batch_size = 32

def generate_answer(texts):
    inputs = tokenizer(texts, 
                     padding=True, 
                     max_length=2048, 
                     padding_side="left",        
                    truncation=True, return_tensors="pt").to("cuda")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024, 
                                 pad_token_id=tokenizer.pad_token_id, 
                                 eos_token_id=tokenizer.eos_token_id, 
                                 do_sample=True, temperature=0.7, top_p=0.9)
        
        answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return answer


In [ ]:
from tqdm import tqdm

all_answers = []
for i in tqdm(range(0, len(prompts), batch_size), desc="Generating answers"):
    batch_prompts = prompts[i:i+batch_size]
    batch_answers = generate_answer(batch_prompts)
    all_answers.extend(batch_answers)

# Print results
for q, a in zip(questions, all_answers):
    # save a csv
    with open("results.csv", "a") as f:
        f.write(f"{q},{a}\n")
    print(f"Question: {q}\nAnswer: {a}\n{'-'*50}\n")

Generating answers:  99%|█████████▊| 69/70 [4:25:09<03:49, 229.95s/it]  